# Probability for Engineers
# Chapter 6: Jointly Distributed Random Variables

---

### Table of Contents
1. [Joint Cumulative Distribution Function (CDF)](#1)
2. [Discrete Joint Distribution — Sock Example](#2)
3. [Marginal and Conditional Distributions (Discrete)](#3)
4. [Continuous Joint Distribution](#4)
5. [Computing Marginal PDFs](#5)
6. [Independent Random Variables](#6)
7. [Sum of Independent RVs — Convolution](#7)
8. [Conditional Distributions: Discrete Case](#8)
9. [Conditional Distributions: Continuous Case](#9)
10. [Verification via Simulation](#10)

In [ ]:
# ─── Required libraries ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from fractions import Fraction
import itertools

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.3
})
DARK_RED = '#8B0000'
print('Libraries loaded.')

<a id='1'></a>
## 1. Joint Cumulative Distribution Function (CDF)

### Definition
Given a pair of random variables (discrete or continuous) $X$ and $Y$,  
the **joint cumulative distribution function** of $X$ and $Y$ is:

$$F_{X,Y}(x, y) = P[X \leq x,\; Y \leq y]$$

Geometric interpretation: the probability that the point $(X, Y)$ lies **to the southwest** of $(x, y)$.

### Key Properties
| Property | Formula |
|---|---|
| Marginal CDF — X | $F_X(x) = F_{X,Y}(x, \infty)$ |
| Marginal CDF — Y | $F_Y(y) = F_{X,Y}(\infty, y)$ |
| Rectangle probability | $P(x_1 < X \leq x_2,\; y_1 < Y \leq y_2) = F(x_2,y_2)+F(x_1,y_1)-F(x_1,y_2)-F(x_2,y_1)$ |
| Joint tail probability | $P(X > x,\; Y > y) = 1 - F_X(x) - F_Y(y) + F_{X,Y}(x,y)$ |

In [ ]:
# ─── Geometric interpretation of the CDF ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: southwest region
ax = axes[0]
ax.set_xlim(-1, 5); ax.set_ylim(-1, 5)
ax.fill_between([-1, 3], [-1, -1], [4, 4], alpha=0.18, color=DARK_RED, label='$P[X\\leq x, Y\\leq y]$')
ax.axvline(3, color=DARK_RED, lw=1.5, ls='--')
ax.axhline(4, color=DARK_RED, lw=1.5, ls='--')
ax.plot(3, 4, 'o', color=DARK_RED, ms=8, zorder=5)
ax.annotate('$(x, y)$', (3, 4), (3.2, 4.2), fontsize=12)
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.set_title('$F_{X,Y}(x,y)$ — Southwest Region')
ax.legend(fontsize=10)

# Right: rectangle probability
ax2 = axes[1]
ax2.set_xlim(-0.5, 5); ax2.set_ylim(-0.5, 5)
x1, x2, y1, y2 = 1, 3.5, 1.5, 4
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, color=DARK_RED, alpha=0.25)
ax2.add_patch(rect)
for xi, ls in zip([x1, x2], ['--', '-']):
    ax2.axvline(xi, color=DARK_RED, lw=1.2, ls=ls)
for yi, ls in zip([y1, y2], ['--', '-']):
    ax2.axhline(yi, color=DARK_RED, lw=1.2, ls=ls)
ax2.text((x1+x2)/2, (y1+y2)/2, '$P(x_1 < X \\leq x_2,\\; y_1 < Y \\leq y_2)$',
         ha='center', va='center', fontsize=9, color='darkred', fontweight='bold')
ax2.set_xlabel('X'); ax2.set_ylabel('Y')
ax2.set_title('Rectangle Probability')
plt.tight_layout()
plt.show()

<a id='2'></a>
## 2. Discrete Joint Distribution — Sock Example

**Problem:** Draw 2 socks at random **without replacement** from a drawer containing 12 colored socks:
- 6 black, 4 white, 2 purple

$B$ = number of black socks drawn, $W$ = number of white socks drawn

Joint PMF:
$$P(B=b,\; W=w) = \frac{\binom{6}{b}\binom{4}{w}\binom{2}{2-b-w}}{\binom{12}{2}}, \quad 0 \leq b, w \leq 2,\; b+w \leq 2$$

In [ ]:
from math import comb

# ─── Joint PMF computation ────────────────────────────────────────────────
total = comb(12, 2)  # = 66

joint_pmf = {}
for b in range(3):
    for w in range(3):
        purple = 2 - b - w
        if 0 <= purple <= 2:
            joint_pmf[(b, w)] = comb(6, b) * comb(4, w) * comb(2, purple) / total
        else:
            joint_pmf[(b, w)] = 0

# ─── Table display ────────────────────────────────────────────────────────
print('=' * 55)
print(f"  Joint PMF  p_{{B,W}}(b,w)    [Total = {total}]")
print('=' * 55)
print(f"{'':>8}  W=0       W=1       W=2    | P(B=b)")
print('-' * 55)

for b in range(3):
    row = [joint_pmf[(b, w)] for w in range(3)]
    fracs = [Fraction(int(p*66), 66) for p in row]
    marginal_b = sum(row)
    print(f"  B={b}   {str(fracs[0]):>8}  {str(fracs[1]):>8}  {str(fracs[2]):>8}  | {Fraction(int(marginal_b*66),66)}")

print('-' * 55)
marginal_w = [sum(joint_pmf[(b, w)] for b in range(3)) for w in range(3)]
fracs_w = [Fraction(int(p*66), 66) for p in marginal_w]
print(f"P(W=w)  {str(fracs_w[0]):>8}  {str(fracs_w[1]):>8}  {str(fracs_w[2]):>8}  | 1")
print('=' * 55)

In [ ]:
# ─── Heat map visualization ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Joint PMF matrix
matrix = np.array([[joint_pmf[(b, w)] for w in range(3)] for b in range(3)])
cmap = LinearSegmentedColormap.from_list('dr', ['white', DARK_RED])

im = axes[0].imshow(matrix, cmap=cmap, vmin=0, vmax=matrix.max(), aspect='auto')
for b in range(3):
    for w in range(3):
        val = Fraction(int(joint_pmf[(b,w)]*66), 66)
        axes[0].text(w, b, str(val), ha='center', va='center', fontsize=11,
                     color='white' if matrix[b,w] > 0.2 else 'black', fontweight='bold')
axes[0].set_xticks([0,1,2]); axes[0].set_xticklabels(['W=0','W=1','W=2'])
axes[0].set_yticks([0,1,2]); axes[0].set_yticklabels(['B=0','B=1','B=2'])
axes[0].set_title('Joint PMF $p_{B,W}(b,w)$', fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Marginal distributions
marg_b = matrix.sum(axis=1)
marg_w = matrix.sum(axis=0)
x = np.arange(3)
axes[1].bar(x - 0.2, marg_b, 0.35, label='P(B=b)', color=DARK_RED, alpha=0.8)
axes[1].bar(x + 0.2, marg_w, 0.35, label='P(W=w)', color='steelblue', alpha=0.8)
for i in range(3):
    axes[1].text(i-0.2, marg_b[i]+0.005, f'{Fraction(int(marg_b[i]*66),66)}', ha='center', fontsize=9, color=DARK_RED)
    axes[1].text(i+0.2, marg_w[i]+0.005, f'{Fraction(int(marg_w[i]*66),66)}', ha='center', fontsize=9, color='steelblue')
axes[1].set_xticks(x); axes[1].set_xticklabels(['0', '1', '2'])
axes[1].set_xlabel('k'); axes[1].set_ylabel('Probability')
axes[1].set_title('Marginal Distributions')
axes[1].legend()
plt.tight_layout()
plt.show()

<a id='3'></a>
## 3. Marginal and Conditional Distributions (Discrete)

### Marginal PMF
$$P(X = x) = \sum_y P(X=x,\; Y=y)$$

### Conditional PMF
$$p_{X|Y}(x|y) = P(X=x \mid Y=y) = \frac{P(X=x,\; Y=y)}{P(Y=y)} = \frac{\text{joint pmf}}{\text{marginal pmf}}$$

In [ ]:
# ─── Conditional distribution: P(B | W=w) ────────────────────────────────
print('Conditional Distribution: P(B = b | W = w)')
print('=' * 50)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for w_val in range(3):
    pW = sum(joint_pmf[(b, w_val)] for b in range(3))
    if pW == 0:
        continue
    cond = {b: joint_pmf[(b, w_val)] / pW for b in range(3)}

    print(f"\nW = {w_val}  [P(W={w_val}) = {Fraction(int(pW*66),66)}]:")
    for b in range(3):
        print(f"  P(B={b} | W={w_val}) = {Fraction(int(cond[b]*pW*66), int(pW*66)).limit_denominator(100)}  ≈ {cond[b]:.4f}")

    axes[w_val].bar(range(3), [cond[b] for b in range(3)],
                    color=DARK_RED, alpha=0.75, edgecolor='white')
    axes[w_val].set_xticks([0,1,2])
    axes[w_val].set_xticklabels(['B=0','B=1','B=2'])
    axes[w_val].set_title(f'$P(B \\mid W={w_val})$', fontweight='bold')
    axes[w_val].set_ylim(0, 1)
    for b in range(3):
        axes[w_val].text(b, cond[b]+0.02, f'{cond[b]:.3f}', ha='center', fontsize=9)

axes[0].set_ylabel('Probability')
plt.suptitle('Conditional Distributions $P(B \\mid W=w)$', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='4'></a>
## 4. Continuous Joint Distribution

### Definition
Random variables $X$ and $Y$ are **jointly continuous** if there exists a function $f(x,y)$ such that:
1. $f(x,y) \geq 0$ — non-negative
2. $\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} f(x,y)\,dx\,dy = 1$

### Textbook Example — Uniform Distribution (Triangular Region)

$$f(x,y) = \begin{cases} c & x \geq 0,\; y \geq 0,\; x+y \leq 3 \\ 0 & \text{otherwise} \end{cases}$$

Normalization: $c \times \frac{3 \times 3}{2} = 1 \implies c = \dfrac{2}{9}$

In [ ]:
# ─── Uniform distribution — triangular region ─────────────────────────────
c = 2/9  # normalization constant

def f_joint(x, y):
    """f(x,y) = 2/9 on the triangular region, 0 elsewhere."""
    return np.where((x >= 0) & (y >= 0) & (x + y <= 3), c, 0.0)

# Normalization verification (numerical)
from scipy.integrate import dblquad
result, error = dblquad(lambda y, x: f_joint(x, y), 0, 3, 0, lambda x: 3-x)
print(f'Normalization check: ∫∫ f(x,y) dx dy = {result:.6f}  (Error: {error:.2e})')
print(f'c = 2/9 = {2/9:.6f}')

# ─── 2D and 3D visualization ──────────────────────────────────────────────
fig = plt.figure(figsize=(13, 5))

# 2D: support region
ax1 = fig.add_subplot(121)
triangle_x = [0, 3, 0, 0]
triangle_y = [0, 0, 3, 0]
ax1.fill(triangle_x, triangle_y, color=DARK_RED, alpha=0.25, label='Support region')
ax1.plot(triangle_x, triangle_y, color=DARK_RED, lw=2)
ax1.text(0.8, 0.8, f'$f(x,y) = c = 2/9$', fontsize=11, color=DARK_RED)
ax1.set_xlim(-0.3, 4); ax1.set_ylim(-0.3, 4)
ax1.set_xlabel('x'); ax1.set_ylabel('y')
ax1.set_title('Support Region: $x\\geq 0,\\; y\\geq 0,\\; x+y\\leq 3$', fontweight='bold')
ax1.legend()
ax1.set_aspect('equal')

# 3D: density surface
ax2 = fig.add_subplot(122, projection='3d')
xv = np.linspace(0, 3, 60)
yv = np.linspace(0, 3, 60)
X2, Y2 = np.meshgrid(xv, yv)
Z2 = f_joint(X2, Y2)
ax2.plot_surface(X2, Y2, Z2, cmap='Reds', alpha=0.85, edgecolor='none')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('f(x,y)')
ax2.set_title('Joint PDF Surface', fontweight='bold')

plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. Computing Marginal PDFs

### Formula
$$f_X(x) = \int_{-\infty}^{\infty} f(x,y)\,dy \qquad f_Y(y) = \int_{-\infty}^{\infty} f(x,y)\,dx$$

### Triangular Region Example
Given $X = x$, the range of $Y$ is $[0, 3-x]$:

$$f_X(x) = \int_0^{3-x} \frac{2}{9}\,dy = \frac{2}{9}(3-x), \quad x \in [0,3]$$

### Additional Example: Computing P(X < Y)
$$P(X < Y) = \iint_{x<y} f(x,y)\,dx\,dy = \int_0^{3/2} \int_x^{3-x} \frac{2}{9}\,dy\,dx = \frac{1}{2}$$

In [ ]:
# ─── Marginal PDFs ────────────────────────────────────────────────────────
def f_X(x):
    """Analytic marginal: f_X(x) = (2/9)(3-x), x in [0,3]"""
    return np.where((x >= 0) & (x <= 3), (2/9)*(3 - x), 0.0)

def f_Y(y):
    """By symmetry: f_Y(y) = f_X(y)"""
    return np.where((y >= 0) & (y <= 3), (2/9)*(3 - y), 0.0)

# Numerical verification
from scipy.integrate import quad
I_fX, _ = quad(f_X, 0, 3)
print(f'∫ f_X(x) dx = {I_fX:.6f}  (should equal 1)')

# P(X < Y) analytic = 1/2
P_XltY, _ = dblquad(lambda y, x: f_joint(x, y), 0, 1.5, lambda x: x, lambda x: 3-x)
print(f'P(X < Y) = {P_XltY:.6f}  (analytic = 0.5)')

# ─── Visualization ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Marginal f_X
xv = np.linspace(-0.5, 3.5, 300)
axes[0].plot(xv, f_X(xv), color=DARK_RED, lw=2.5)
axes[0].fill_between(np.linspace(0,3,200), f_X(np.linspace(0,3,200)), alpha=0.2, color=DARK_RED)
axes[0].set_title('Marginal PDF: $f_X(x) = \\frac{2}{9}(3-x)$', fontweight='bold')
axes[0].set_xlabel('x'); axes[0].set_ylabel('$f_X(x)$')
axes[0].set_xlim(-0.3, 3.5)

# Marginal f_Y
yv = np.linspace(-0.5, 3.5, 300)
axes[1].plot(yv, f_Y(yv), color='steelblue', lw=2.5)
axes[1].fill_between(np.linspace(0,3,200), f_Y(np.linspace(0,3,200)), alpha=0.2, color='steelblue')
axes[1].set_title('Marginal PDF: $f_Y(y) = \\frac{2}{9}(3-y)$', fontweight='bold')
axes[1].set_xlabel('y'); axes[1].set_ylabel('$f_Y(y)$')
axes[1].set_xlim(-0.3, 3.5)

# P(X<Y) region
ax3 = axes[2]
xv2 = np.linspace(0, 3, 200)
ax3.fill(triangle_x, triangle_y, color='lightgray', alpha=0.4, label='Full support')
# X < Y region (above x=y line, inside triangle)
xx = np.linspace(0, 1.5, 100)
ax3.fill_between(xx, xx, 3-xx, color=DARK_RED, alpha=0.4, label='$X < Y$')
ax3.plot([0, 3], [0, 3], 'k--', lw=1.5, label='$x = y$')
ax3.plot(triangle_x, triangle_y, color='gray', lw=1.5)
ax3.text(0.3, 1.8, '$P(X<Y) = 1/2$', fontsize=11, color=DARK_RED, fontweight='bold')
ax3.set_xlim(-0.2, 3.5); ax3.set_ylim(-0.2, 3.5)
ax3.set_xlabel('x'); ax3.set_ylabel('y')
ax3.set_title('$P(X < Y)$ Region', fontweight='bold')
ax3.legend(fontsize=9)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()

<a id='6'></a>
## 6. Independent Random Variables

### Definition
Random variables $X$ and $Y$ are **independent** if for any real sets $A, B \subset \mathbb{R}$:
$$P(X \in A,\; Y \in B) = P(X \in A) \cdot P(Y \in B)$$

$X$ and $Y$ are independent **if and only if**:

| Case | Condition |
|---|---|
| General | $F_{X,Y}(x,y) = F_X(x) \cdot F_Y(y)$ |
| Discrete | $p_{X,Y}(x,y) = p_X(x) \cdot p_Y(y)$ |
| Continuous | $f_{X,Y}(x,y) = f_X(x) \cdot f_Y(y)$ |

### Practical Test
If $f_{X,Y}(x,y)$ can be factored as $g(x)\cdot h(y)$ → independent!

> **Note:** The uniform distribution over the triangular region is **not independent** — because the support region cannot be separated due to the constraint $x+y \leq 3$.

In [ ]:
# ─── Independence test: is the triangular example independent? ────────────
print('Independence Test — f(x,y) = f_X(x) * f_Y(y)?')
print('=' * 55)

test_pts = [(0.5, 0.5), (1.0, 1.0), (0.5, 1.5), (1.5, 0.5)]
for (x0, y0) in test_pts:
    joint   = f_joint(x0, y0)
    product = f_X(x0) * f_Y(y0)
    eq = '✓ equal' if np.isclose(joint, product, atol=1e-9) else '✗ different'
    print(f'  x={x0}, y={y0}: f(x,y)={joint:.5f},  fX*fY={product:.5f}  → {eq}')

print()
print('Conclusion: The uniform distribution on the triangular region is NOT INDEPENDENT.')
print('Reason: The support x+y≤3 links the values of X and Y.')

# ─── Example of an independent distribution: unit square ─────────────────
print()
print('Independent Example: f(x,y) = 1, (x,y) ∈ [0,1]×[0,1]')
def f_ind(x, y):
    return np.where((0 <= x) & (x <= 1) & (0 <= y) & (y <= 1), 1.0, 0.0)

for (x0, y0) in [(0.3, 0.7), (0.5, 0.2), (0.8, 0.9)]:
    joint   = f_ind(x0, y0)
    fX_val  = 1.0 if 0 <= x0 <= 1 else 0.0
    fY_val  = 1.0 if 0 <= y0 <= 1 else 0.0
    product = fX_val * fY_val
    eq = '✓ equal' if np.isclose(joint, product) else '✗ different'
    print(f'  x={x0}, y={y0}: f(x,y)={joint:.1f},  fX*fY={product:.1f}  → {eq}')

In [ ]:
# ─── Meeting Problem (Textbook Example) ───────────────────────────────────
# A male and female student each arrive independently at a uniform random
# time between 12:00 and 13:00. P(|X - Y| > 10) = ?

print('Meeting Problem')
print('=' * 55)
print('X, Y ~ Uniform(0, 60) minutes, independent')
print('P(|X - Y| > 10) = ?')
print()

# Analytic: band area = 60^2 - 2*(1/2)*(50^2) = 3600 - 2500 = 1100
# P(|X-Y| <= 10) = 1100/3600 = 11/36
# P(|X-Y| > 10) = 25/36
analytic = 25/36
print(f'Analytic result: P(|X-Y| > 10) = 25/36 ≈ {analytic:.5f}')

# Monte Carlo simulation
np.random.seed(42)
N = 1_000_000
X_sim = np.random.uniform(0, 60, N)
Y_sim = np.random.uniform(0, 60, N)
P_sim = np.mean(np.abs(X_sim - Y_sim) > 10)
print(f'Monte Carlo ({N:,} samples): P(|X-Y| > 10) ≈ {P_sim:.5f}')
print(f'Error: {abs(P_sim - analytic):.5f}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Geometric
ax = axes[0]
square = plt.Polygon([[0,0],[60,0],[60,60],[0,60]], fill=True,
                      facecolor='lightblue', edgecolor='gray', alpha=0.5)
ax.add_patch(square)
# |X-Y| ≤ 10 band
band_x = [0, 50, 60, 60, 10, 0]
band_y = [10, 60, 60, 50, 0, 0]
band = plt.Polygon(list(zip(band_x, band_y)), facecolor=DARK_RED, alpha=0.3)
ax.add_patch(band)
ax.plot([0,60],[10,70],'k--', lw=1, label='$Y=X+10$')
ax.plot([0,60],[-10,50],'k:', lw=1, label='$Y=X-10$')
ax.plot([0,60],[0,60],'gray', lw=0.8, ls='-', alpha=0.6)
ax.set_xlim(0,60); ax.set_ylim(0,60)
ax.set_xlabel('X (minutes)'); ax.set_ylabel('Y (minutes)')
ax.set_title(f'P(|X-Y|>10) = 25/36 ≈ {analytic:.3f}\n(red: |X-Y|≤10 band)', fontweight='bold')
ax.legend(fontsize=9)

# Simulation scatter
n_plot = 3000
idx = np.random.choice(N, n_plot, replace=False)
mask = np.abs(X_sim[idx] - Y_sim[idx]) > 10
axes[1].scatter(X_sim[idx][~mask], Y_sim[idx][~mask], s=2, color='steelblue', alpha=0.5, label='|X-Y|≤10')
axes[1].scatter(X_sim[idx][mask],  Y_sim[idx][mask],  s=2, color=DARK_RED, alpha=0.5, label='|X-Y|>10')
axes[1].set_xlabel('X (minutes)'); axes[1].set_ylabel('Y (minutes)')
axes[1].set_title(f'Monte Carlo Simulation (n={n_plot:,})', fontweight='bold')
axes[1].legend(markerscale=5, fontsize=9)

plt.tight_layout()
plt.show()

<a id='7'></a>
## 7. Sum of Independent RVs — Convolution

### Continuous Convolution Formula
If $X$ and $Y$ are independent:
$$f_{X+Y}(z) = \int_{-\infty}^{\infty} f_X(x)\, f_Y(z-x)\,dx$$

### Known Results
| X | Y | X + Y |
|---|---|---|
| $\mathcal{N}(\mu_1, \sigma_1^2)$ | $\mathcal{N}(\mu_2, \sigma_2^2)$ | $\mathcal{N}(\mu_1+\mu_2, \sigma_1^2+\sigma_2^2)$ |
| $\text{Poi}(\lambda_1)$ | $\text{Poi}(\lambda_2)$ | $\text{Poi}(\lambda_1+\lambda_2)$ |
| $\text{Bin}(n_1,p)$ | $\text{Bin}(n_2,p)$ | $\text{Bin}(n_1+n_2,p)$ |

### Textbook Example: Triangular Distribution
$X, Y \sim \text{Unif}(0,1)$ independent → $Z = X+Y$ follows a triangular distribution:
$$f_{X+Y}(z) = \begin{cases} z & 0 < z \leq 1 \\ 2-z & 1 < z < 2 \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
# ─── Sum of Two Uniform(0,1) → Triangular Distribution ───────────────────
def f_triangle(z):
    z = np.asarray(z, dtype=float)
    out = np.zeros_like(z)
    out = np.where((z > 0) & (z <= 1), z, out)
    out = np.where((z > 1) & (z < 2), 2 - z, out)
    return out

# Monte Carlo verification
np.random.seed(0)
N = 500_000
Z_sim = np.random.uniform(0, 1, N) + np.random.uniform(0, 1, N)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Analytic PDF
zv = np.linspace(-0.2, 2.2, 400)
axes[0].plot(zv, f_triangle(zv), color=DARK_RED, lw=2.5, label='Analytic $f_{X+Y}$')
axes[0].fill_between(np.linspace(0,2,400), f_triangle(np.linspace(0,2,400)),
                     alpha=0.2, color=DARK_RED)
axes[0].set_title('Triangular Distribution (Analytic)', fontweight='bold')
axes[0].set_xlabel('z = x + y'); axes[0].set_ylabel('$f_{X+Y}(z)$')
axes[0].legend()

# Simulation histogram
axes[1].hist(Z_sim, bins=80, density=True, color='steelblue', alpha=0.7,
             edgecolor='white', lw=0.3, label='Simulation')
axes[1].plot(zv, f_triangle(zv), 'r-', lw=2.5, label='Analytic')
axes[1].set_title(f'Monte Carlo Verification (n={N:,})', fontweight='bold')
axes[1].set_xlabel('z'); axes[1].legend()

# Sum of normals
mu1, sig1 = 2, 1
mu2, sig2 = 3, 1.5
Z_norm = np.random.normal(mu1, sig1, N) + np.random.normal(mu2, sig2, N)
zn = np.linspace(-4, 14, 400)
f_sum = stats.norm.pdf(zn, mu1+mu2, np.sqrt(sig1**2+sig2**2))
axes[2].hist(Z_norm, bins=80, density=True, color='steelblue', alpha=0.7,
             edgecolor='white', lw=0.3, label='Simulation')
axes[2].plot(zn, f_sum, 'r-', lw=2.5,
             label=f'$\\mathcal{{N}}({mu1+mu2}, {sig1**2+sig2**2:.1f})$')
axes[2].set_title(f'$\\mathcal{{N}}({mu1},{sig1}^2)+\\mathcal{{N}}({mu2},{sig2}^2)$\n→ Normal (closed form)', fontweight='bold')
axes[2].set_xlabel('z'); axes[2].legend()

plt.tight_layout()
plt.show()

print(f'Triangular distribution mean: E[Z] = {Z_sim.mean():.4f}  (expected: 1.0)')
print(f'Triangular distribution variance: Var[Z] = {Z_sim.var():.4f}  (expected: 1/6 ≈ {1/6:.4f})')

<a id='8'></a>
## 8. Conditional Distributions: Discrete Case

### Definition
For discrete random variables $X$ and $Y$:
$$p_{X|Y}(x|y) = P(X=x \mid Y=y) = \frac{p(x,y)}{p_Y(y)}, \quad p_Y(y) > 0$$

### Textbook Example — Ball Drawing
A bag contains 5 white and 8 red balls; 3 balls are drawn without replacement.  
$X_i = 1$ if the $i$-th white ball is selected, $0$ otherwise ($i = 1, 2$).

In [ ]:
# ─── Ball drawing example: joint PMF ─────────────────────────────────────
total_balls = comb(13, 3)

# p(x1, x2): joint PMF
# x1=1 → white ball 1 selected, x2=1 → white ball 2 selected
# p(0,0): neither selected — choose 3 from remaining 11
# p(1,1): both selected — choose 1 from remaining 11
# p(0,1)=p(1,0): exactly one selected — choose 2 from remaining 11, with 1 from the 2 white balls

p00 = comb(11, 3) * comb(2, 0) / total_balls
p11 = comb(11, 1) * comb(2, 2) / total_balls
p01 = comb(11, 2) * comb(2, 1) / total_balls
p10 = p01  # symmetry

joint_balls = {(0,0): p00, (0,1): p01, (1,0): p10, (1,1): p11}

print('Joint PMF p(x1, x2):')
print('=' * 45)
for (x1, x2), p in joint_balls.items():
    f = Fraction(int(p * total_balls), int(total_balls)).limit_denominator(100)
    print(f'  p({x1},{x2}) = {f} = {p:.5f}')

# Conditional distributions
pX2_1 = sum(joint_balls[(x1, 1)] for x1 in [0,1])
pX2_0 = sum(joint_balls[(x1, 0)] for x1 in [0,1])

print()
print(f'Marginal: P(X2=1) = {Fraction(int(pX2_1*total_balls), int(total_balls))}')
print(f'Marginal: P(X2=0) = {Fraction(int(pX2_0*total_balls), int(total_balls))}')

print()
print('(1) P(X1 = x1 | X2=1):')
for x1 in [0, 1]:
    cond = joint_balls[(x1, 1)] / pX2_1
    f = Fraction(int(joint_balls[(x1,1)]*total_balls), int(pX2_1*total_balls)).limit_denominator(20)
    print(f'  P(X1={x1} | X2=1) = {f} ≈ {cond:.4f}')

print()
print('(2) P(X1 = x1 | X2=0):')
for x1 in [0, 1]:
    cond = joint_balls[(x1, 0)] / pX2_0
    f = Fraction(int(joint_balls[(x1,0)]*total_balls), int(pX2_0*total_balls)).limit_denominator(20)
    print(f'  P(X1={x1} | X2=0) = {f} ≈ {cond:.4f}')

In [ ]:
# ─── Visualization: conditional PMFs ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

cases = [
    (None,  [p00+p10, p01+p11], 'Unconditional $P(X_1)$', 'gray'),
    (1,     [joint_balls[(0,1)]/pX2_1, joint_balls[(1,1)]/pX2_1], '$P(X_1 \\mid X_2=1)$', DARK_RED),
    (0,     [joint_balls[(0,0)]/pX2_0, joint_balls[(1,0)]/pX2_0], '$P(X_1 \\mid X_2=0)$', 'steelblue'),
]

for ax, (cond_val, probs, title, color) in zip(axes, cases):
    ax.bar([0, 1], probs, color=color, alpha=0.75, edgecolor='white')
    for i, p in enumerate(probs):
        ax.text(i, p + 0.01, f'{p:.3f}', ha='center', fontsize=11)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['$X_1=0$', '$X_1=1$'])
    ax.set_ylim(0, 1.0); ax.set_ylabel('Probability')
    ax.set_title(title, fontweight='bold')

plt.suptitle('Conditional PMF Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Conditional Distributions: Continuous Case

### Definition
Given the joint density $f(x,y)$ of $X$ and $Y$, the conditional PDF of $X$ given $Y = y$ (for $f_Y(y) > 0$) is:

$$f_{X|Y}(x|y) = \frac{f(x,y)}{f_Y(y)}$$

### Textbook Example
$$f(x,y) = \frac{12}{5}x(2-x-y), \quad 0 < x < 1,\; 0 < y < 1$$

$$f_{X|Y}(x|y) = \frac{x(2-x-y)}{\int_0^1 x(2-x-y)\,dx} = \frac{6x(2-x-y)}{4-3y}$$

In [ ]:
# ─── Continuous conditional distribution ─────────────────────────────────
def f_joint2(x, y):
    """f(x,y) = (12/5)*x*(2-x-y), 0<x<1, 0<y<1"""
    return np.where((0 < x) & (x < 1) & (0 < y) & (y < 1),
                    (12/5) * x * (2 - x - y), 0.0)

# Normalization check
norm, _ = dblquad(lambda y, x: float(f_joint2(x, y)), 0, 1, 0, 1)
print(f'Normalization: ∫∫ f(x,y) dx dy = {norm:.6f}  (should equal 1)')

def f_Y2(y):
    """Marginal: f_Y(y) = ∫₀¹ f(x,y) dx = (12/5)*(2/3 - y/2)"""
    return np.where((0 < y) & (y < 1), (12/5) * (2/3 - y/2), 0.0)

def f_cond(x, y):
    """Conditional: f_{X|Y}(x|y) = 6x(2-x-y)/(4-3y)"""
    return np.where((0 < x) & (x < 1), 6*x*(2-x-y)/(4-3*y), 0.0)

# ─── Conditional distributions for different values of y ──────────────────
y_vals = [0.1, 0.3, 0.5, 0.7, 0.9]
xv = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Left: joint PDF
xg, yg = np.meshgrid(np.linspace(0,1,80), np.linspace(0,1,80))
zg = f_joint2(xg, yg)
im = axes[0].contourf(xg, yg, zg, levels=20, cmap='Reds')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Joint PDF $f(x,y)$', fontweight='bold')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# Middle: marginal f_Y
yv = np.linspace(0, 1, 300)
axes[1].plot(yv, f_Y2(yv), color=DARK_RED, lw=2.5)
axes[1].fill_between(yv, f_Y2(yv), alpha=0.2, color=DARK_RED)
axes[1].set_title('Marginal PDF $f_Y(y) = \\frac{12}{5}\\left(\\frac{2}{3}-\\frac{y}{2}\\right)$', fontweight='bold')
axes[1].set_xlabel('y'); axes[1].set_ylabel('$f_Y(y)$')

# Right: conditional distributions
colors = plt.cm.RdPu(np.linspace(0.3, 0.9, len(y_vals)))
for y0, col in zip(y_vals, colors):
    fc = f_cond(xv, y0)
    area, _ = quad(lambda x: f_cond(x, y0), 0, 1)
    axes[2].plot(xv, fc, color=col, lw=2, label=f'y={y0} (∫={area:.3f})')

axes[2].set_title('Conditional PDF $f_{X|Y}(x|y)$', fontweight='bold')
axes[2].set_xlabel('x'); axes[2].set_ylabel('$f_{X|Y}(x|y)$')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\nConditional PDF normalization check:')
for y0 in y_vals:
    area, _ = quad(lambda x: float(f_cond(x, y0)), 0, 1)
    print(f'  y={y0}: ∫₀¹ f_{{X|Y}}(x|{y0}) dx = {area:.6f}')

<a id='10'></a>
## 10. Verification via Simulation — Comprehensive Example

Let us simulate the joint distribution on the triangular region to verify theoretical results:  
$f(x,y) = 2/9$, $x \geq 0,\; y \geq 0,\; x+y \leq 3$

In [ ]:
# ─── Rejection Sampling ───────────────────────────────────────────────────
np.random.seed(123)
N_target = 200_000

# Sample from [0,3]x[0,3] box; accept those satisfying x+y≤3
candidates = np.random.uniform(0, 3, (N_target * 3, 2))
mask = candidates[:, 0] + candidates[:, 1] <= 3
samples = candidates[mask][:N_target]
X_samp, Y_samp = samples[:, 0], samples[:, 1]

print(f'Sample size: {len(X_samp):,}')

# ─── Verification table ───────────────────────────────────────────────────
print('\n' + '=' * 60)
print(f'{"Quantity":<30} {"Theoretical":>12} {"Simulation":>12}')
print('=' * 60)

checks = [
    ('E[X]',         1.0,       X_samp.mean()),
    ('E[Y]',         1.0,       Y_samp.mean()),
    ('P(X < Y)',     0.5,       np.mean(X_samp < Y_samp)),
    ('P(X+Y > 2)',   5/9,       np.mean(X_samp + Y_samp > 2)),
    ('P(X < 1)',     5/9,       np.mean(X_samp < 1)),
]

for name, theory, sim in checks:
    print(f'{name:<30} {theory:>12.5f} {sim:>12.5f}')
print('=' * 60)

# ─── Visualization ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Scatter
n_show = 5000
axes[0].scatter(X_samp[:n_show], Y_samp[:n_show], s=1.5, alpha=0.4, color=DARK_RED)
axes[0].plot([0,3],[3,0],'k-',lw=2)
axes[0].set_title(f'Sample Points (n={n_show:,})', fontweight='bold')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
axes[0].set_aspect('equal')

# X marginal
xv = np.linspace(0, 3, 300)
axes[1].hist(X_samp, bins=60, density=True, color='steelblue', alpha=0.6, label='Simulation')
axes[1].plot(xv, f_X(xv), color=DARK_RED, lw=2.5, label='$f_X(x)=(2/9)(3-x)$')
axes[1].set_title('X Marginal Distribution', fontweight='bold')
axes[1].set_xlabel('x'); axes[1].legend(fontsize=9)

# Y marginal
axes[2].hist(Y_samp, bins=60, density=True, color='steelblue', alpha=0.6, label='Simulation')
axes[2].plot(xv, f_Y(xv), color=DARK_RED, lw=2.5, label='$f_Y(y)=(2/9)(3-y)$')
axes[2].set_title('Y Marginal Distribution', fontweight='bold')
axes[2].set_xlabel('y'); axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 📝 Summary Table

| Concept | Discrete | Continuous |
|---|---|---|
| **Joint distribution** | $p_{X,Y}(x,y) = P(X=x, Y=y)$ | $f_{X,Y}(x,y) \geq 0$, $\iint = 1$ |
| **Marginal** | $p_X(x) = \sum_y p(x,y)$ | $f_X(x) = \int f(x,y)\,dy$ |
| **Conditional** | $p_{X|Y}(x|y) = p(x,y)/p_Y(y)$ | $f_{X|Y}(x|y) = f(x,y)/f_Y(y)$ |
| **Independence** | $p(x,y) = p_X(x)p_Y(y)$ | $f(x,y) = f_X(x)f_Y(y)$ |
| **Sum (convolution)** | — | $f_{X+Y}(z) = \int f_X(x)f_Y(z-x)\,dx$ |

---
*Probability for Engineers — Haydar Kilic*